# 🤖 Week 3: Model Development & Experiment Tracking
## Telco Customer Churn — Baseline Models + MLflow

**Goal:** Train three baseline classifiers, compare them using AUC / F1 / precision / recall, and track every experiment in MLflow so results are fully reproducible.

### Agenda
1. Load the processed pipeline output (from Week 2)
2. Train Logistic Regression, Random Forest, XGBoost
3. Compare metrics + ROC curves
4. Inspect confusion matrices
5. Open the MLflow UI to explore runs

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve

from src.data.pipeline import run_full_pipeline
from src.models.train import train_all_baselines
from src.models.evaluate import compute_metrics, print_report, log_confusion_matrix

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup complete ✓')

---
## 1. Load & Prepare Data

> **Prerequisite:** Download `WA_Fn-UseC_-Telco-Customer-Churn.csv` into `data/raw/`  
> See the README for Kaggle download instructions.

In [ ]:
X_train, X_test, y_train, y_test, pipe = run_full_pipeline()

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Churn rate — Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}')

---
## 2. Train All Baseline Models

Each model is automatically logged to MLflow with:
- Hyperparameters
- Metrics (AUC, F1, accuracy, precision, recall)
- Confusion matrix artefact
- Serialised model artefact

In [ ]:
results = train_all_baselines(
    X_train, y_train,
    X_test,  y_test,
    log_to_mlflow=True,
)

# Attach test data for ROC curves
for r in results.values():
    r['X_test'] = X_test
    r['y_test']  = y_test

---
## 3. Metrics Comparison

In [ ]:
rows = []
for name, r in results.items():
    row = {'Model': name}
    row.update(r['metrics'])
    rows.append(row)

metrics_df = pd.DataFrame(rows).set_index('Model').sort_values('roc_auc', ascending=False)
display(metrics_df.style
    .background_gradient(cmap='RdYlGn', subset=['roc_auc','f1','accuracy'])
    .format('{:.4f}')
    .set_caption('Baseline Model Comparison'))

### Metric bar chart

In [ ]:
metrics_to_plot = ['roc_auc', 'f1', 'accuracy', 'precision', 'recall']
ax = metrics_df[metrics_to_plot].plot(
    kind='bar', figsize=(11, 5), ylim=(0, 1.05),
    color=sns.color_palette('deep', len(metrics_to_plot))
)
ax.set_xlabel('')
ax.set_ylabel('Score')
ax.set_title('Baseline Model Metrics Comparison', fontweight='bold')
ax.legend(loc='lower right')
ax.axhline(0.5, color='red', linestyle='--', alpha=0.4, label='Random baseline')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 4. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#4C72B0', '#DD8452', '#55A868']

for (name, r), color in zip(results.items(), colors):
    model = r['model']
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = r['metrics']['roc_auc']
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})', color=color, lw=2)

ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Baseline Models', fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, r) in zip(axes, results.items()):
    model = r['model']
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices (threshold = 0.5)', y=1.02, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Full Classification Reports

In [ ]:
for name, r in results.items():
    print_report(r['model'], X_test, y_test, model_name=name)

---
## 7. Open the MLflow UI

Run this in a terminal (separate from the notebook):

```bash
cd "/Users/leonberger/Documents/comp sci /DataSci"
venv/bin/mlflow ui
```

Then open: **[http://localhost:5000](http://localhost:5000)**

You'll see all three runs under the `churn-baseline` experiment with:
- Side-by-side metric comparison charts
- Downloadable model artefacts
- Confusion matrix images
- Full parameter logs

---
## ✅ Week 3 Summary

| Model | Typical AUC | Notes |
|-------|-------------|-------|
| Logistic Regression | ~0.84 | Fast, interpretable, great baseline |
| Random Forest | ~0.83 | Robust, handles non-linearity |
| XGBoost | ~0.85 | Usually best out of the box |

> **Churn is imbalanced (~27% positive)** — AUC and F1 are more meaningful than raw accuracy.

**Next: Week 4 — Hyperparameter Tuning + SHAP Model Explainability**
- Use Optuna to find optimal XGBoost hyperparameters
- Generate SHAP waterfall, beeswarm, and summary plots
- Understand *why* customers churn, not just *who* will churn